# LoRA fine-tune — FoxSchool intent classification

**Runtime:** T4 GPU · **Model:** Llama 3.2 1B Instruct

In [10]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers peft "trl>=1.6.0" accelerate datasets "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.8 MB/s eta 0:00:00


In [4]:
from huggingface_hub import login

login()  # token: huggingface.co/settings/tokens

import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, get_peft_model

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# Llama 1B в fp16 ~2 GB
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [11]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [12]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train_alpaca.jsonl", split="train")

PROMPT = """### Instruction:
{}

### Input:
{}

### Response:
{}"""


def format_example(example):
    text = PROMPT.format(
        example["instruction"],
        example["input"],
        example["output"],
    ) + tokenizer.eos_token
    return {"text": text}


dataset = dataset.map(format_example)
print(dataset[0]["text"][:300])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

### Instruction:
Classify this FoxSchool support ticket. Reply with only one label: billing, bug, how-to, refund, other.

### Input:
I'm interested in upgrading my plan, but I’m unsure about the additional charges that will apply.

### Response:
billing<|eot_id|>


In [13]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./outputs",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        fp16=False,
        bf16=False,
        optim="adamw_torch",
        save_strategy="no",
        max_length=512,
        loss_type="nll",
    ),
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,1.703848
20,0.702113
30,0.548970


TrainOutput(global_step=39, training_loss=0.8737769493689904, metrics={'train_runtime': 78.1588, 'train_samples_per_second': 7.677, 'train_steps_per_second': 0.499, 'total_flos': 203680084770816.0, 'train_loss': 0.8737769493689904, 'entropy': 0.5414392807904411, 'mean_token_accuracy': 0.8636762236847597, 'num_tokens': 32610.0, 'epoch': 3.0})

In [14]:
test_text = "I was charged twice for the Beginner plan"
prompt = PROMPT.format(
    "Classify this FoxSchool support ticket. Reply with only one label: billing, bug, how-to, refund, other.",
    test_text,
    "",
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Instruction:
Classify this FoxSchool support ticket. Reply with only one label: billing, bug, how-to, refund, other.

### Input:
I was charged twice for the Beginner plan

### Response:
billing###################{###


In [15]:
import shutil

from google.colab import files

SAVE_DIR = "foxschool-intent-lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

shutil.make_archive("foxschool-intent-lora", "zip", SAVE_DIR)
files.download("foxschool-intent-lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>